# Tutorial 01 — Getting Started: Core Abstractions

This notebook is a **bottom-up tour** of every core building block in lllm:

| Section | Concepts covered |
|---------|------------------|
| 1 | Environment setup, providers, `Tactic.quick()` |
| 2 | `Agent` — identity and lifecycle |
| 3 | `Dialog` — the three-step pattern |
| 4 | Multi-turn conversations |
| 5 | Multiple dialogs and `switch()` |
| 6 | Forking and closing dialogs |
| 7 | The `Message` object |
| 8 | `Prompt` — templates and variables |
| 9 | Structured output: XML tags, Markdown blocks, signal tags |
| 10 | Pydantic structured output |
| 11 | Prompt inheritance with `extend()` |
| 12 | Context window management |
| 13 | Building an `Agent` from scratch (low-level) |

---
## Section 0 — Environment Setup

lllm uses [LiteLLM](https://github.com/BerriAI/litellm) under the hood, which means it can talk to **any major LLM provider** through a unified interface. The only thing you need is the right API key.

### Supported providers

| Provider | Model string example | Environment variable |
|----------|---------------------|----------------------|
| OpenAI | `gpt-4o`, `gpt-4o-mini` | `OPENAI_API_KEY` |
| Anthropic | `claude-opus-4-6`, `claude-sonnet-4-6`, `claude-haiku-4-5-20251001` | `ANTHROPIC_API_KEY` |
| Mistral | `mistral/mistral-large` | `MISTRAL_API_KEY` |
| Any other | see LiteLLM docs | provider-specific |

### Setting keys

The recommended approach is a `.env` file at the project root (never commit it):

```
# .env
OPENAI_API_KEY=sk-...
ANTHROPIC_API_KEY=sk-ant-...
```

Then load it before running any cells:

In [ ]:
# Install dotenv if needed: pip install python-dotenv
from dotenv import load_dotenv
import os

# load_dotenv() searches for .env starting from the current directory upward
load_dotenv(dotenv_path="../.env", override=False)

# Verify at least one key is present
has_openai = bool(os.getenv("OPENAI_API_KEY"))
has_anthropic = bool(os.getenv("ANTHROPIC_API_KEY"))

if not (has_openai or has_anthropic):
    raise EnvironmentError(
        "No API key found. Set OPENAI_API_KEY or ANTHROPIC_API_KEY in your .env file."
    )

print(f"OpenAI key present  : {has_openai}")
print(f"Anthropic key present: {has_anthropic}")

# Choose a default model for this notebook
DEFAULT_MODEL = "gpt-4o-mini" if has_openai else "claude-haiku-4-5-20251001"
print(f"\nUsing model: {DEFAULT_MODEL}")

In [ ]:
# Install lllm if needed
# %pip install lllm-core python-dotenv

import sys
sys.path.insert(0, "..")

import lllm
from lllm import Agent, Prompt, Tactic
from lllm.invokers import build_invoker
from helpers.display import print_response, print_section, print_dialog

print(f"lllm version: {lllm.__version__ if hasattr(lllm, '__version__') else 'installed'}")

---
## Section 1 — The Zero-Config Entry Point: `Tactic.quick()`

`Tactic.quick()` is the fastest path to a working LLM call.  
No config files, no subclassing, no provider setup — just a model name and a message.

Internally it:
1. Builds a `Prompt` from the system prompt string
2. Builds a LiteLLM invoker
3. Creates an `Agent`
4. Calls `agent.open("chat")` → `agent.receive(query)` → `agent.respond()`
5. Returns the `Message`

There are three calling modes:

In [ ]:
# Mode 1: one-shot — pass a query, get back a Message
response = Tactic.quick(
    "What is the capital of France?",
    model=DEFAULT_MODEL,
)
print("Type  :", type(response))
print("Reply :", response.content)

In [ ]:
# Mode 2: one-shot with custom system prompt
response = Tactic.quick(
    "What is 7 times 6?",
    model=DEFAULT_MODEL,
    system_prompt="You are a grumpy math teacher who always sighs before answering.",
)
print(response.content)

In [ ]:
# Mode 3: return the Agent for follow-up turns
response, agent = Tactic.quick(
    "Name three popular Python web frameworks.",
    model=DEFAULT_MODEL,
    return_agent=True,
)
print("First reply:")
print(response.content)
print()

# Continue the conversation using the returned agent
agent.receive("Which one is best for building REST APIs?")
followup = agent.respond()
print("Follow-up:")
print(followup.content)

In [ ]:
# Mode 4: get an agent without sending a query yet
agent = Tactic.quick(
    system_prompt="You are a friendly Python tutor. Keep examples short.",
    model=DEFAULT_MODEL,
)
print("Agent name:", agent.name)
print("Agent model:", agent.model)

---
## Section 2 — The `Agent` Object

An `Agent` is an **LLM identity**. It owns:
- A **name** (used in logs and dialog ownership)
- A **system prompt** (`Prompt` object)
- A **model** (any LiteLLM model string)
- An **invoker** (the transport layer to the LLM API)
- Zero or more **dialogs** (named conversation histories)

Agents do not store results from different conversations in a shared pool — every dialog is independent and named.

---
## Section 3 — The Dialog: `open → receive → respond`

Every interaction with an agent follows the same three-step pattern:

```
agent.open("alias")          # 1. Create (or resume) a named dialog
agent.receive("message")     # 2. Append a user message
response = agent.respond()   # 3. Call the LLM, append its reply, return Message
```

The **alias** is a human-readable name for the conversation. You can hold multiple dialogs on the same agent simultaneously and switch between them.

In [ ]:
agent = Tactic.quick(
    system_prompt="You are a concise assistant. Answer in one or two sentences.",
    model=DEFAULT_MODEL,
)

# Step 1: open a dialog named "chat"
agent.open("chat")

# Step 2: add a user message
agent.receive("What is a closure in Python?")

# Step 3: invoke the LLM and get the response
msg = agent.respond()
print(msg.content)

---
## Section 4 — Multi-Turn Conversations

Each `receive → respond` cycle appends to the **same dialog**.  
The full message history is sent to the LLM on every turn, giving the model context across the conversation.

In [ ]:
agent = Tactic.quick(
    system_prompt="You are a Python expert. Be brief.",
    model=DEFAULT_MODEL,
)

agent.open("lesson")

# Turn 1
agent.receive("Explain Python generators in one sentence.")
r1 = agent.respond()
print("[Turn 1]", r1.content)

# Turn 2 — the model remembers what it just said
agent.receive("Now show me a minimal code example.")
r2 = agent.respond()
print("\n[Turn 2]", r2.content)

# Turn 3 — referencing the example from turn 2
agent.receive("What happens if I call next() after the generator is exhausted?")
r3 = agent.respond()
print("\n[Turn 3]", r3.content)

In [ ]:
# Inspect the full dialog
dialog = agent.current_dialog
print(f"Dialog has {len(dialog.messages)} messages:\n")
for msg in dialog.messages:
    role = str(msg.role).split(".")[-1].upper()
    preview = str(msg.content)[:80].replace("\n", " ")
    print(f"  [{role}] {preview}")

In [ ]:
# Dialog overview — condensed snapshot
print(dialog.overview())

In [ ]:
# Dialog head and tail
print("head (first):", str(dialog.head.content)[:80])
print("tail (last) :", str(dialog.tail.content)[:80])

---
## Section 5 — Multiple Dialogs and `switch()`

One agent can hold **several named dialogs simultaneously** and switch between them freely.  
This is useful when a single agent is being used for multiple independent topics, or when you want to keep a main thread while exploring a tangent.

In [ ]:
agent = Tactic.quick(
    system_prompt="You are a knowledgeable assistant.",
    model=DEFAULT_MODEL,
)

# Open two independent dialogs
agent.open("astronomy")
agent.receive("Tell me one sentence about black holes.")
r_astro = agent.respond()
print("[astronomy]", r_astro.content)

agent.open("cooking")
agent.receive("Tell me one sentence about sourdough.")
r_cook = agent.respond()
print("[cooking]  ", r_cook.content)

# Switch back to astronomy and continue
agent.switch("astronomy")
agent.receive("What is Hawking radiation?")
r_astro2 = agent.respond()
print("\n[astronomy continued]")
print(r_astro2.content)

# Switch back to cooking — it still remembers the previous context
agent.switch("cooking")
agent.receive("What is a starter?")
r_cook2 = agent.respond()
print("\n[cooking continued]")
print(r_cook2.content)

In [ ]:
# Each dialog is fully independent
for name in ["astronomy", "cooking"]:
    agent.switch(name)
    d = agent.current_dialog
    print(f"{name}: {len(d.messages)} messages, cost so far: {d.cost}")

---
## Section 6 — Forking and Closing Dialogs

**Forking** creates a child dialog that shares history up to the fork point, then evolves independently.  
The parent dialog is **never modified**. This is perfect for exploring "what if" questions from the same starting context.

**Closing** removes a dialog from the agent and returns the `Dialog` object for archiving.

In [ ]:
agent = Tactic.quick(
    system_prompt="You are a creative writing assistant.",
    model=DEFAULT_MODEL,
)

# Establish shared context in the main dialog
agent.open("main")
agent.receive("Our story is set in a futuristic city where robots and humans coexist.")
agent.respond()  # model acknowledges the premise

print("After shared setup, main dialog has", len(agent.current_dialog.messages), "messages\n")

# Fork A: explore a dark, dystopian direction
agent.fork("main", "dark_branch")
agent.receive("Write a tense opening paragraph where the robots are rebelling.")
dark_reply = agent.respond()
print("=== Dark Branch ===")
print(dark_reply.content[:400])

# Fork B: explore a hopeful, utopian direction
agent.switch("main")  # go back to the unchanged parent
agent.fork("main", "light_branch")
agent.receive("Write a warm opening paragraph where robots and humans celebrate together.")
light_reply = agent.respond()
print("\n=== Light Branch ===")
print(light_reply.content[:400])

In [ ]:
# Verify: the main dialog was never modified
agent.switch("main")
print("Main dialog message count:", len(agent.current_dialog.messages))
print("(Still just the premise setup — no branched messages)")

In [ ]:
# Close a branch to free it from the agent (returns the Dialog object)
archived = agent.close("dark_branch")
print("Archived dialog type:", type(archived))
print("Archived message count:", len(archived.messages))

---
## Section 7 — The `Message` Object

Every `agent.respond()` call returns a `Message`. It carries more than just text:

In [ ]:
agent = Tactic.quick(
    system_prompt="You are a concise assistant.",
    model=DEFAULT_MODEL,
)
agent.open("demo")
agent.receive("What are the three laws of robotics?")
msg = agent.respond()

print("msg.content         :", msg.content[:120], "...")
print("msg.role            :", msg.role)
print("msg.name            :", msg.name)
print("msg.usage           :", msg.usage)
print("msg.cost            :", msg.cost)
print("msg.is_function_call:", msg.is_function_call)
print("msg.parsed          :", msg.parsed)  # None unless a parser is attached

In [ ]:
# Token cost breakdown
cost = msg.cost
if cost:
    print("Prompt tokens    :", cost.prompt_tokens)
    print("Completion tokens:", cost.completion_tokens)
    print("Dollar cost      : ${:.6f}".format(cost.cost))

---
## Section 8 — The `Prompt` Object

A `Prompt` is the blueprint for an agent's behavior. It bundles:
- A **template** string (with optional `{variable}` placeholders)
- An optional **parser** for structured output
- An optional **parser** for output validation
- An optional list of **tools** the agent can call

The `path` is a unique identifier used in the resource registry and logs.

In [ ]:
# Minimal prompt
p = Prompt(
    path="tutorial/greeter",
    prompt="You are a friendly assistant. Greet the user warmly.",
)
print("Path          :", p.path)
print("Template      :", p.prompt)
print("Template vars :", p.template_vars)  # no variables in this one

In [ ]:
# Prompt with template variables
p_templated = Prompt(
    path="tutorial/analyst",
    prompt=(
        "You are a senior analyst specializing in {domain}. "
        "Today's date is {date}. "
        "Always cite your sources and quantify claims where possible."
    ),
)

print("Template vars:", p_templated.template_vars)

# Validate before rendering — useful for catching missing variables early
missing = p_templated.validate_args({"domain": "AI"})
print("Missing vars :", missing)  # ['date']

# Render by calling the prompt object
rendered = p_templated(domain="macroeconomics", date="2026-03-17")
print("\nRendered:\n", rendered)

In [ ]:
# Use a templated prompt in an agent
# When you call agent.open(alias, prompt_args={...}), the args are forwarded to the prompt
from lllm.invokers import build_invoker

invoker = build_invoker({"invoker": "litellm"})

analyst = Agent(
    name="analyst",
    system_prompt=p_templated,
    model=DEFAULT_MODEL,
    llm_invoker=invoker,
)

# prompt_args are forwarded to the system prompt's __call__ when the dialog opens
analyst.open("session", prompt_args={"domain": "renewable energy", "date": "2026-03-17"})
analyst.receive("What are the three biggest trends in solar power right now?")
r = analyst.respond()
print(r.content[:500])

---
## Section 9 — Structured Output: Parsers

Attaching a `DefaultTagParser` to a `Prompt` tells the agent to extract structured data from the LLM's response. Three formats are supported:

- **XML tags** — `<tag>content</tag>` blocks
- **Markdown code blocks** — ` ```language ... ``` ` blocks
- **Signal tags** — bare `<TAG>` tokens that act as boolean flags

All parsed data ends up in `message.parsed`.

In [ ]:
from lllm.core.prompt import DefaultTagParser

# --- 9a. XML tag parsing ---
extractor_prompt = Prompt(
    path="tutorial/extractor",
    prompt=(
        "Extract named entities from the given text. "
        "Return your answer using these XML tags:\n"
        "<people>Comma-separated person names</people>\n"
        "<organizations>Comma-separated organization names</organizations>\n"
        "<locations>Comma-separated location names</locations>"
    ),
    parser=DefaultTagParser(
        xml_tags=["people", "organizations", "locations"],
        required_xml_tags=["people"],  # raises ParseError if missing
    ),
)

invoker = build_invoker({"invoker": "litellm"})
extractor = Agent(
    name="extractor",
    system_prompt=extractor_prompt,
    model=DEFAULT_MODEL,
    llm_invoker=invoker,
)

text = (
    "Elon Musk announced that Tesla would expand its Gigafactory in Berlin. "
    "The news was covered by Reuters and The Wall Street Journal."
)

extractor.open("extract")
extractor.receive(text)
msg = extractor.respond()

print("Raw response:")
print(msg.content)
print("\nParsed:")
import json
print(json.dumps(msg.parsed, indent=2))

In [ ]:
# Access specific extracted entities
parsed = msg.parsed
print("People       :", parsed["xml_tags"].get("people", []))
print("Organizations:", parsed["xml_tags"].get("organizations", []))
print("Locations    :", parsed["xml_tags"].get("locations", []))

In [ ]:
# --- 9b. Markdown code block parsing ---
coder_prompt = Prompt(
    path="tutorial/coder",
    prompt=(
        "You are a Python coding assistant. "
        "Always put your code solution inside a ```python code block."
    ),
    parser=DefaultTagParser(
        md_tags=["python"],
        required_md_tags=["python"],
    ),
)

coder = Agent(
    name="coder",
    system_prompt=coder_prompt,
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
)

coder.open("coding")
coder.receive("Write a function that checks if a string is a palindrome.")
code_msg = coder.respond()

print("Extracted Python code:")
blocks = code_msg.parsed["md_tags"].get("python", [])
for i, block in enumerate(blocks):
    print(f"\n--- Block {i+1} ---")
    print(block)

In [ ]:
# --- 9c. Signal tags (boolean flags) ---
classifier_prompt = Prompt(
    path="tutorial/sentiment_classifier",
    prompt=(
        "Classify the sentiment of the given review. "
        "If the sentiment is positive, include <POSITIVE> anywhere in your response. "
        "Briefly explain your classification."
    ),
    parser=DefaultTagParser(signal_tags=["POSITIVE"]),
)

classifier = Agent(
    name="classifier",
    system_prompt=classifier_prompt,
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
)

reviews = [
    "Absolutely loved it — best product I've bought this year!",
    "It broke after two days. Very disappointed.",
    "It's okay, does the job but nothing special.",
]

for review in reviews:
    classifier.open("classify")
    classifier.receive(review)
    m = classifier.respond()
    is_positive = m.parsed["signal_tags"]["POSITIVE"]
    print(f"Review  : {review[:60]}")
    print(f"Positive: {is_positive}")
    print(f"Reason  : {m.content[:120]}\n")
    classifier.close("classify")

---
## Section 10 — Pydantic Structured Output

For JSON / structured output via the model's native support, pass `format=MyModel` to the `Prompt`.  
The LLM returns valid JSON that lllm deserializes into a Pydantic instance, available at `message.parsed`.

> **Note:** Requires model support — works with `gpt-4o`, `gpt-4o-mini`, and recent Anthropic models.

In [ ]:
from pydantic import BaseModel, Field


class ArticleSummary(BaseModel):
    headline: str = Field(description="A concise headline for the article")
    key_points: list[str] = Field(description="Three to five key takeaways")
    sentiment: str = Field(description="Overall tone: positive, neutral, or negative")
    word_count_estimate: int = Field(description="Estimated word count of the original text")


summary_prompt = Prompt(
    path="tutorial/summarizer",
    prompt="Summarize the given article. Return a structured JSON response.",
    format=ArticleSummary,
)

summarizer = Agent(
    name="summarizer",
    system_prompt=summary_prompt,
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
)

article = """
OpenAI has announced GPT-5, their most powerful model to date. The model demonstrates
significant improvements in reasoning, coding, and multimodal understanding. Early
benchmarks show it outperforming all previous models on standard tests. The company plans
to roll out access gradually through the API and ChatGPT. Competitors including Anthropic
and Google are expected to respond with their own updates. Analysts see this as a pivotal
moment in the AI arms race.
"""

summarizer.open("summarize")
summarizer.receive(article.strip())
msg = summarizer.respond()

summary: ArticleSummary = msg.parsed
print("Type          :", type(summary))
print("Headline      :", summary.headline)
print("Key points    :")
for pt in summary.key_points:
    print(f"  - {pt}")
print("Sentiment     :", summary.sentiment)
print("Word estimate :", summary.word_count_estimate)

In [ ]:
# Pydantic lets you easily serialize to dict or JSON
print(summary.model_dump_json(indent=2))

In [ ]:
# Richer schema with nested models and validators
from pydantic import field_validator


class ProductReviewAnalysis(BaseModel):
    sentiment: str = Field(description="positive, neutral, or negative")
    score: float = Field(description="Sentiment score between 0.0 (negative) and 1.0 (positive)")
    pros: list[str] = Field(default_factory=list, description="Up to 3 positive aspects")
    cons: list[str] = Field(default_factory=list, description="Up to 3 negative aspects")
    recommend: bool

    @field_validator("score")
    @classmethod
    def validate_score(cls, v: float) -> float:
        if not 0.0 <= v <= 1.0:
            raise ValueError("score must be between 0.0 and 1.0")
        return round(v, 2)


review_prompt = Prompt(
    path="tutorial/review_analyzer",
    prompt="Analyze the product review and return a structured JSON object.",
    format=ProductReviewAnalysis,
)

review_agent = Agent(
    name="review_analyzer",
    system_prompt=review_prompt,
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
)

reviews_to_analyze = [
    "Fantastic keyboard! The tactile switches are perfect and build quality is excellent.",
    "Battery life is terrible. Barely lasts 4 hours. Returned it immediately.",
    "Decent chair, comfortable for short sessions but lacks lumbar support for long days.",
]

for review in reviews_to_analyze:
    review_agent.open("analyze")
    review_agent.receive(review)
    m = review_agent.respond()
    a: ProductReviewAnalysis = m.parsed
    print(f"Review   : {review[:60]}...")
    print(f"Sentiment: {a.sentiment} (score={a.score})  recommend={a.recommend}")
    print(f"Pros: {a.pros}")
    print(f"Cons: {a.cons}\n")
    review_agent.close("analyze")

---
## Section 11 — Prompt Inheritance with `extend()`

`extend()` copies all fields of a base prompt and applies overrides. A new `path` is mandatory.  
This is useful for creating specialized variants of a shared base without copy-pasting.

In [ ]:
base_prompt = Prompt(
    path="tutorial/base_writer",
    prompt="You are a professional writer working on {topic}. Be clear and concise.",
)

# Specialized variants
tech_prompt = base_prompt.extend(
    path="tutorial/tech_writer",
    prompt=(
        "You are a senior technical writer working on {topic}. "
        "Use precise terminology. Always include code examples where relevant."
    ),
)

blog_prompt = base_prompt.extend(
    path="tutorial/blog_writer",
    prompt=(
        "You are an engaging blog writer working on {topic}. "
        "Use a conversational tone. Aim for an 8th-grade reading level."
    ),
)

print("Base :", base_prompt.prompt)
print("Tech :", tech_prompt.prompt)
print("Blog :", blog_prompt.prompt)

In [ ]:
# Compare outputs from the two specialized prompts on the same topic
topic = "Python decorators"

for prompt_obj, label in [(tech_prompt, "Tech"), (blog_prompt, "Blog")]:
    a = Agent(
        name=label.lower(),
        system_prompt=prompt_obj,
        model=DEFAULT_MODEL,
        llm_invoker=build_invoker({"invoker": "litellm"}),
    )
    a.open("write", prompt_args={"topic": topic})
    a.receive(f"Write a two-sentence intro about {topic}.")
    r = a.respond()
    print(f"[{label}] {r.content}\n")

---
## Section 12 — Context Window Management

Long conversations eventually exceed the model's context limit.  
`DefaultContextManager` prunes the dialog automatically **before each LLM call** without mutating the stored history.

What it preserves:
- The **first message** (system prompt) — always
- The **most recent messages** — kept from the tail inward until the token budget is exhausted
- A **5 000-token safety buffer** below the context limit
- The **oldest kept message** is character-truncated with `[...earlier content truncated...]` if it only partially fits

In [ ]:
from lllm.core.dialog import DefaultContextManager

# Create a context manager with a tight cap (useful for cheap models)
cm = DefaultContextManager(
    model_name=DEFAULT_MODEL,
    max_tokens=4096,  # keep conversation history within 4k tokens
)

agent_with_cm = Agent(
    name="long_conv",
    system_prompt=Prompt(path="tutorial/cm_demo", prompt="You are a helpful assistant."),
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
    context_manager=cm,
)

agent_with_cm.open("chat")

# Simulate a growing conversation
questions = [
    "What is quantum computing?",
    "What is a qubit?",
    "How does superposition work?",
    "What makes quantum computers faster for certain problems?",
]

for q in questions:
    agent_with_cm.receive(q)
    r = agent_with_cm.respond()
    print(f"Q: {q}")
    print(f"A: {r.content[:150]}...\n")

print("\nFull dialog message count:", len(agent_with_cm.current_dialog.messages))
print("(The context manager ensures each LLM call stays under the token budget)")

In [ ]:
# Writing a custom context management policy
from lllm.core.dialog import ContextManager, Dialog

class RollingWindowManager(ContextManager):
    """Keep only the system prompt + the last N exchanges."""
    name = "rolling_window"

    def __init__(self, model_name: str, keep_last: int = 6):
        self.model_name = model_name
        self.keep_last = keep_last

    def __call__(self, dialog: Dialog) -> Dialog:
        if len(dialog.messages) <= self.keep_last + 1:  # +1 for system prompt
            return dialog  # still within the window, no-op
        # Fork: keep the first message (system) and the last keep_last messages
        return dialog.fork(last_n=self.keep_last, first_k=1)


rolling_cm = RollingWindowManager(model_name=DEFAULT_MODEL, keep_last=4)

rolling_agent = Agent(
    name="rolling",
    system_prompt=Prompt(path="tutorial/rolling", prompt="You are a helpful assistant."),
    model=DEFAULT_MODEL,
    llm_invoker=build_invoker({"invoker": "litellm"}),
    context_manager=rolling_cm,
)

rolling_agent.open("chat")
for q in questions:
    rolling_agent.receive(q)
    rolling_agent.respond()

print("Total stored messages:", len(rolling_agent.current_dialog.messages))
print("(The LLM sees at most 4 recent messages + the system prompt each call)")

---
## Section 13 — Building an `Agent` from Scratch (Low-Level)

Everything in this notebook used `Tactic.quick()` to skip boilerplate.  
Here is what that convenience method does internally — useful when you need full control.

In [ ]:
from lllm import Agent, Prompt
from lllm.invokers import build_invoker

# 1. Create a Prompt
system_prompt = Prompt(
    path="my_project/assistant/system",
    prompt="You are a knowledgeable assistant. Be direct and precise.",
)

# 2. Build an invoker (transport layer to the LLM API)
#    The "litellm" invoker supports every provider lllm ships with
invoker = build_invoker({"invoker": "litellm"})

# 3. Create the Agent
agent = Agent(
    name="assistant",          # used in logs and dialog ownership
    system_prompt=system_prompt,
    model=DEFAULT_MODEL,
    llm_invoker=invoker,
    max_interrupt_steps=5,     # max tool calls per respond() cycle
    max_exception_retry=3,     # max retries on ParseError
)

# 4. Use it with the three-step pattern
agent.open("chat")
agent.receive("What is lllm and what problem does it solve?")
msg = agent.respond()
print(msg.content[:400])

---
## Summary

| Concept | Key API |
|---------|----------|
| One-shot call | `Tactic.quick(query, model=...)` → `Message` |
| Get an agent | `Tactic.quick(system_prompt=..., model=...)` → `Agent` |
| Three-step pattern | `agent.open(alias)` → `agent.receive(msg)` → `agent.respond()` |
| Multi-turn | `agent.receive()` + `agent.respond()` on same dialog |
| Multiple dialogs | `agent.open("b")` then `agent.switch("a")` |
| Fork | `agent.fork("src", "dest")` |
| Close | `agent.close("alias")` → returns `Dialog` |
| Message fields | `.content`, `.role`, `.cost`, `.parsed` |
| Prompt | `Prompt(path=..., prompt=...)` |
| Template vars | `{variable}` in template; `prompt(var=val)` to render |
| XML/MD/signal parsing | `DefaultTagParser(xml_tags=[...], md_tags=[...], signal_tags=[...])` |
| Pydantic output | `Prompt(format=MyModel)` → `msg.parsed` is a `MyModel` instance |
| Prompt inheritance | `base.extend(path=..., ...)` |
| Context management | `Agent(context_manager=DefaultContextManager(...))` |

**Next:** [02_tactics_and_agentic_systems.ipynb](02_tactics_and_agentic_systems.ipynb) — coordinate multiple agents with `Tactic`.